In [1]:
import os
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from dateutil import parser

# Ensure downloaded these resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

# Path to the text folder
text_folder_path = r"C:\Users\nici_\OneDrive\Desktop\Masterarbeit\Factsheets\Text"

# Define stopwords and lemmatizer
custom_stopwords = {
    'der', 'die', 'das', 'den', 'des', 'le', 'la', 'les', 'de', 'et', 'und', 'davon', 'per', 'name', 
    'postcode', 'address', 'phone', 'company', 'ag', 'am',  'namen', 'bbb', 'inhalt', 'total', 'anhang,'
    'management', 'phone', 'telefon', 'telefax', 'email', 'weiterhin', 'com', 'whg', 'senior', 'seite',
    'u', 'ab', 'seit', 'mehr', 'company_1', 'company_2','company_3', 'company_4', 'company_5','company_6',   
 
  # Pronouns and articles not in NLTK
    'ich', 'du', 'er', 'wir', 'ihr', 'uns', 'mein', 'dein', 'sein', 'unser', 'euer', 'deren', 'dessen',
    
    # Common conjunctions and prepositions not in NLTK
    'obwohl', 'trotz', 'denn', 'doch', 'da', 'weder', 'noch', 'falls', 'sowie', 'entweder', 'trotzdem',
    'daher', 'deswegen', 'hinzu', 'daraufhin', 'hinweg', 'hierbei',
    
    # Additional time-related words
    'heute', 'morgen', 'gestern', 'jetzt', 'sofort', 'bald', 'manchmal', 'früh', 'spät', 'vorher', 'danach',
    'eben', 'gerade', 'schon', 'bereits', 'weiterhin', 'zudem', 'fort', 'sofort',
    
    # Other frequent words
    'jedoch', 'vielleicht', 'beispielsweise', 'beispiel', 'eigentlich', 'alle', 'meist', 'sagen', 'machen', 
    'gut', 'schlecht', 'wenig', 'viel', 'vielen', 'ebenfalls', 'nein', 'ja', 'teil', 'sehr', 'hier', 'dort',
    'auch', 'beim', 'viele', 'alle', 'mehr', 'dann', 'immer', 'einige', 'jemand', 'niemand', 'mehrere', 'solche'
    

    'ersten', 'zweiten', 'leicht', 'kopie', 'gegenüber', 'insbesondere', 'flexiblen', 'stark', 'aufgrund', 'isin',
    # Common English stopwords
    'therefore', 'thus', 'furthermore', 'additionally', 'also', 'however', 'could', 'would', 'might', 'must',
    'should', 'including', 'etc', 'either', 'neither', 'one', 'two', 'three', 'first', 'second', 'third', 'many',
}
stop_words = set(stopwords.words('english')) | set(stopwords.words('german')) | set(stopwords.words('french')) | custom_stopwords
lemmatizer = WordNetLemmatizer()

# Define month names for regex-based date extraction in multiple languages
month_names = [
    "january", "february", "march", "april", "may", "june", "july", "august", "september", "october", "november", "december",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
    "janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre", "novembre", "décembre",
    "janv", "févr", "avr", "août", "sept", "oct", "nov", "déc",
    "januar", "februar", "märz", "april", "mai", "juni", "juli", "august", "september", "oktober", "november", "dezember",
    "jan", "feb", "mär", "apr", "jun", "jul", "aug", "sep", "okt", "nov", "dez"
]
month_stopwords = set(month_names)  # For filtering months post date extraction

# Date pattern to match various date formats in English, French, and German
date_pattern = (
    r"\b(?:\d{1,2}[-/\s]?)?"
    + r"(?:" + "|".join(month_names) + r")[-/\s]?"
    + r"(?:\d{2,4})?\b"
    + r"|\b\d{1,2}[-/\s]\d{1,2}[-/\s]\d{2,4}\b"
)

# Define a mapping for normalizing similar terms
word_mapping = {
    "aktienähnliche": "aktien",

}

# Function to standardize and clean text
def standardize_text(text):
    text = re.sub(r'\s+', ' ', text).strip()
    aligned_text = '\n'.join(line.lstrip() for line in text.splitlines())
    return aligned_text

# Function to extract dates using regex and parse them
def extract_dates(text):
    dates = []
    for match in re.finditer(date_pattern, text, flags=re.IGNORECASE):
        try:
            parsed_date = parser.parse(match.group(), fuzzy=True, dayfirst=True)
            dates.append(parsed_date.isoformat())
        except (ValueError, parser.ParserError):
            continue
    return dates if dates else None

# Function to preprocess text by removing stopwords, numbers, and performing lemmatization
def preprocess_text_for_topic_modeling(text):
    text = text.lower()
    
    # Remove all numbers, including years
    text = re.sub(r'\b\d+\b', '', text)
    
    # Remove non-alphanumeric characters
    text = re.sub(r'\W', ' ', text)

    processed_words = []
    for word in text.split():
        # Remove short words (length <= 2), stopwords, and words with minimal meaning
        if len(word) > 2 and word not in stop_words and word not in month_stopwords:
            word = lemmatizer.lemmatize(word)
            # Apply word mapping if applicable
            word = word_mapping.get(word, word)
            processed_words.append(word)

    return ' '.join(processed_words)



text_data = {}
for file in os.listdir(text_folder_path):
    if file.endswith(".txt"):
        with open(os.path.join(text_folder_path, file), 'r', encoding='utf-8') as f:
            raw_text = f.read()
            standardized_text = standardize_text(raw_text)  # Standardize and align text
            
            # Extract dates with years
            extracted_dates = extract_dates(standardized_text)
            
            # Preprocess text for topic modeling (without years or numbers)
            preprocessed_text_for_modeling = preprocess_text_for_topic_modeling(standardized_text)
            
            # Store both preprocessed text and extracted dates
            text_data[file] = {
                "dates": extracted_dates,
                "processed_text": preprocessed_text_for_modeling
            }

# Display extracted dates and processed text examples
for file, content in list(text_data.items())[:3]:  # Display first 3 for brevity
    print(f"\nFile: {file}")
    print(f"Extracted Dates: {content['dates']}")
    print(f"Preprocessed Text:\n{content['processed_text'][:500]}")  # Show first 500 characters of preprocessed text
    
# Collect all processed texts into a list for topic modeling
documents = [content["processed_text"] for content in text_data.values()]



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nici_\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nici_\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nici_\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!



File: 20140930_[COMPANY_2]_10011435-001_350565_001001_page_1.txt
Extracted Dates: ['2014-09-05T00:00:00']
Preprocessed Text:
reporting kundeninformationen stiftung altersrücktritt bauhauptgewerbe far mandat kundenanlageprofil referenzwährung chf betreuungsart custody anlagestrategie balanced ansprechpartner marco kälin mail mail www

File: 20140930_[COMPANY_2]_10011435-001_350565_001001_page_10.txt
Extracted Dates: ['2014-09-05T00:00:00', '2014-11-05T00:00:00', '2013-11-05T00:00:00', '2014-09-05T00:00:00', '2024-01-14T00:00:00', '2024-02-14T00:00:00', '2024-11-14T00:00:00', '2024-04-14T00:00:00', '2024-11-14T00:00:00', '2024-06-14T00:00:00', '2024-07-14T00:00:00', '2024-08-14T00:00:00', '2024-09-14T00:00:00', '2024-01-14T00:00:00', '2024-02-14T00:00:00', '2024-11-14T00:00:00', '2024-04-14T00:00:00', '2024-11-14T00:00:00', '2024-06-14T00:00:00', '2024-07-14T00:00:00', '2024-08-14T00:00:00', '2024-09-14T00:00:00']
Preprocessed Text:
performanceverlauf reporting erstellt performance datu

In [2]:
# Uninstall gensim and numpy
%pip uninstall -y gensim numpy

# Install compatible versions for gensim (4.3.3) and numpy (1.26.4)
%pip install numpy==1.26.4 gensim==4.3.3


from gensim import corpora
from gensim.models import LdaModel

# Collect all processed texts into a list for topic modeling
documents = [content["processed_text"] for content in text_data.values()]


Found existing installation: gensim 4.3.3
Uninstalling gensim-4.3.3:
  Successfully uninstalled gensim-4.3.3
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blis 1.0.1 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
thinc 8.3.2 requires numpy<2.1.0,>=2.0.0; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Using cached gensim-4.3.3-cp311-cp311-win_amd64.whl.metadata (8.2 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
Using cached gensim-4.3.3-cp311-cp311-win_amd64.whl (24.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Create a dictionary and a corpus from the processed text data
dictionary = corpora.Dictionary(doc.split() for doc in documents)
corpus = [dictionary.doc2bow(doc.split()) for doc in documents]

# Train an LDA model
num_topics = 5  # Define the number of topics you want
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42, passes=10)

# Display the topics with the most significant words
print("\nTop words in each topic:")
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx}: {topic}")



Top words in each topic:
Topic 0: 0.014*"quartal" + 0.007*"dollar" + 0.007*"prozent" + 0.006*"aktien" + 0.005*"usa" + 0.004*"anleihen" + 0.004*"renditen" + 0.004*"fonds" + 0.004*"deutlich" + 0.004*"staatsanleihen"
Topic 1: 0.012*"informationen" + 0.010*"msci" + 0.008*"verfügung" + 0.008*"dokument" + 0.007*"gestellt" + 0.007*"daten" + 0.006*"personen" + 0.006*"management" + 0.005*"performance" + 0.005*"soweit"
Topic 2: 0.027*"performance" + 0.019*"anlage" + 0.016*"obligationen" + 0.013*"wertveränderungen" + 0.011*"aktien" + 0.011*"vermögensausweis" + 0.011*"portfolio" + 0.010*"vermögensübersicht" + 0.010*"erträge" + 0.010*"detailpositionen"
Topic 3: 0.223*"chf" + 0.028*"devisenkurs" + 0.027*"obligationen" + 0.026*"veränd" + 0.020*"anlage" + 0.019*"detailpositionen" + 0.018*"aktien" + 0.018*"performance" + 0.017*"anhang" + 0.016*"vermögensausweis"
Topic 4: 0.156*"usd" + 0.082*"eur" + 0.032*"aktien" + 0.020*"veränd" + 0.019*"chf" + 0.018*"gbp" + 0.018*"devisenkurs" + 0.017*"gesundheitswe

In [4]:
%pip install pyLDAvis
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# Prepare the visualization
pyLDAvis.enable_notebook()  # Use enable_notebook() if in Jupyter; otherwise, use show()
lda_vis = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(lda_vis)


In [5]:
%pip install spacy



  Using cached numpy-2.0.2-cp311-cp311-win_amd64.whl.metadata (59 kB)
Using cached numpy-2.0.2-cp311-cp311-win_amd64.whl (15.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
pyarrow 15.0.2 requires numpy<2,>=1.16.6, but you have numpy 2.0.2 which is incompatible.


In [6]:
import spacy.cli

# Download German and English language models
spacy.cli.download("de_core_news_sm")
spacy.cli.download("en_core_web_sm")


✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
nlp_de = spacy.load("de_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")

# Function to preprocess text by removing all parts of speech except nouns and verbs
def preprocess_text_nouns_verbs(text, lang="de"):
    # Select the appropriate spaCy model based on language
    nlp = nlp_de if lang == "de" else nlp_en
    doc = nlp(text.lower())
    
    # Keep only nouns and verbs
    processed_words = [token.lemma_ for token in doc if token.pos_ in {"NOUN", "VERB"}]
    return ' '.join(processed_words)

# Apply the preprocessing function to each document in the list
processed_texts_de = [preprocess_text_nouns_verbs(doc, lang="de") for doc in documents]

# Display the processed texts
for i, processed_text in enumerate(processed_texts_de[:5], start=1):  # Display first 5 for brevity
    print(f"Processed Document {i}:\n{processed_text}\n")


Processed Document 1:
Stiftung altersrücktreten Bauhauptgewerbe far Mandat Anlagestrategie Ansprechpartner

Processed Document 2:
Performanceverlauf erstellen portfolioweren Portfolio differenz differenz Rendite Performanceberechnung portfolioweren Anhang

Processed Document 3:
erstellen differenz Liquidität Tie Tie Performanceberechnung Information Anhang

Processed Document 4:
Vermögensstruktur Strategie abweichung¹ liquidität exposur Strategie Performance Anhang

Processed Document 5:
Vermögensstruktur erstellen Liquidität obligationen marktweren Strategie exposur Strategie information Anhang



In [8]:
tokenized_texts = [text.split() for text in processed_texts_de]


In [9]:
from gensim import corpora

# Create a dictionary and corpus
dictionary = corpora.Dictionary(tokenized_texts)
corpus = [dictionary.doc2bow(text) for text in tokenized_texts]


In [10]:
from gensim.models import LdaModel

# Set the number of topics
num_topics = 5  
# Train the LDA model
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, passes=10, random_state=42)


In [11]:
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx + 1}: {topic}")


Topic 1: 0.047*"Anhang" + 0.047*"Aktie" + 0.047*"Marktwert" + 0.046*"Marktpreis" + 0.046*"Einstandspreis" + 0.043*"vermögensübersicht" + 0.042*"devisenkurs" + 0.041*"Preis" + 0.041*"Performance" + 0.041*"obligationen"
Topic 2: 0.025*"Quartal" + 0.013*"Dollar" + 0.013*"Aktie" + 0.012*"Bank" + 0.010*"Bewertung" + 0.009*"Kurs" + 0.009*"Prozent" + 0.008*"Wert" + 0.007*"Performance" + 0.007*"Jahr"
Topic 3: 0.028*"Information" + 0.024*"stellen" + 0.023*"Verfügung" + 0.019*"Dokument" + 0.017*"daten" + 0.013*"Person" + 0.011*"Land" + 0.010*"schäden" + 0.010*"Unternehmen" + 0.010*"Management"
Topic 4: 0.015*"Währung" + 0.011*"erstellen" + 0.011*"Wert" + 0.011*"Anhang" + 0.011*"Aktie" + 0.009*"Beschreibung" + 0.009*"Quartal" + 0.008*"anlag" + 0.007*"Duration" + 0.007*"Marchzinsen"
Topic 5: 0.079*"Aktie" + 0.031*"Eur" + 0.029*"Performance" + 0.023*"eur" + 0.018*"Anhang" + 0.018*"Industrie" + 0.016*"Allokation" + 0.015*"finanzwesen" + 0.015*"Anlage" + 0.014*"vermögensübersicht"


In [12]:
# Get the dominant topic for each document
document_topics = [lda_model.get_document_topics(bow) for bow in corpus]

# Find the dominant topic and its weight for each document
dominant_topics = [(max(doc, key=lambda x: x[1])) for doc in document_topics]

# Display dominant topics for the first few documents
for i, (topic_num, weight) in enumerate(dominant_topics[:5]):
    print(f"Document {i+1}: Dominant Topic {topic_num + 1} with weight {weight:.2f}")


Document 1: Dominant Topic 1 with weight 0.90
Document 2: Dominant Topic 4 with weight 0.93
Document 3: Dominant Topic 4 with weight 0.80
Document 4: Dominant Topic 5 with weight 0.91
Document 5: Dominant Topic 4 with weight 0.60


In [13]:
# Set the number of top words to display per topic
top_words = 10

for idx, topic in lda_model.print_topics(num_topics=-1, num_words=top_words):
    print(f"\nTopic {idx + 1}:")
    for word, weight in lda_model.show_topic(idx, top_words):
        print(f"{word}: {weight:.4f}")



Topic 1:
Anhang: 0.0473
Aktie: 0.0472
Marktwert: 0.0470
Marktpreis: 0.0457
Einstandspreis: 0.0457
vermögensübersicht: 0.0433
devisenkurs: 0.0425
Preis: 0.0415
Performance: 0.0414
obligationen: 0.0409

Topic 2:
Quartal: 0.0253
Dollar: 0.0129
Aktie: 0.0126
Bank: 0.0117
Bewertung: 0.0105
Kurs: 0.0093
Prozent: 0.0090
Wert: 0.0081
Performance: 0.0074
Jahr: 0.0072

Topic 3:
Information: 0.0283
stellen: 0.0241
Verfügung: 0.0235
Dokument: 0.0193
daten: 0.0172
Person: 0.0129
Land: 0.0111
schäden: 0.0104
Unternehmen: 0.0100
Management: 0.0098

Topic 4:
Währung: 0.0146
erstellen: 0.0109
Wert: 0.0109
Anhang: 0.0106
Aktie: 0.0105
Beschreibung: 0.0094
Quartal: 0.0088
anlag: 0.0085
Duration: 0.0073
Marchzinsen: 0.0072

Topic 5:
Aktie: 0.0788
Eur: 0.0308
Performance: 0.0285
eur: 0.0230
Anhang: 0.0184
Industrie: 0.0178
Allokation: 0.0158
finanzwesen: 0.0153
Anlage: 0.0148
vermögensübersicht: 0.0140


In [15]:
# Number of top documents to display per topic
top_docs_per_topic = 3

# Dictionary to store top documents for each topic
top_docs = {i: [] for i in range(num_topics)}

# Populate top_docs with document weights
for doc_index, doc_topics in enumerate(document_topics):
    for topic_num, weight in doc_topics:
        top_docs[topic_num].append((weight, doc_index))

# Sort and select the top documents per topic
for topic_num, docs in top_docs.items():
    sorted_docs = sorted(docs, key=lambda x: x[0], reverse=True)[:top_docs_per_topic]
    print(f"\nTop documents for Topic {topic_num + 1}:")
    for weight, doc_index in sorted_docs:
        print(f"Document {doc_index + 1} with weight {weight:.2f}:")
        print(documents[doc_index][:300])  # Display the first 300 characters of the document



Top documents for Topic 1:
Document 445 with weight 0.98:
performance vermögensübersicht detailpositionen anhang vermögensausweis portfolio aktien aktien anlage aktien usd bezeichnung einstandspreis marktpreis veränd preis marktwert chf anzahl branche devisenkurs devisenkurs veränd marchzins chf gesundheitswesen usd us0028241000 usd abbvie inc gesundheitsw
Document 379 with weight 0.98:
performance vermögensübersicht detailpositionen anhang vermögensausweis mandat aktien aktien anlage aktien usd bezeichnung einstandspreis marktpreis veränd preis marktwert chf anzahl branche devisenkurs devisenkurs veränd marchzins chf gesundheitswesen usd us0028241000 usd abbvie inc gesundheitswese
Document 412 with weight 0.98:
performance vermögensübersicht detailpositionen anhang vermögensausweis mandat aktien aktien anlage aktien usd bezeichnung einstandspreis marktpreis veränd preis marktwert chf anzahl branche devisenkurs devisenkurs veränd marchzins chf gesundheitswesen usd us0028241000 usd abb